# MSEB: Custom Encoder Registration
This notebook provides the technical requirements for implementing the `MultiModalEncoder` interface. 

In MSEB, an **"Encoder"** is a broad concept. It takes multimodal inputs (text, audio) and outputs either:
1. **Representations:** Dense vector embeddings (e.g., standard text or audio embedding models).
2. **Direct Predictions:** Direct text answers or outputs (e.g., LLMs performing a generative task).

To participate in the benchmark, your model must be wrapped in a class that adheres to the MSEB execution flow and be formally registered. This guide walks through creating, registering, and verifying both types. 

> 💡 **Tip:** For real-world production examples of both types, we highly recommend exploring the `mseb/encoders/` directory in the repository (e.g., `hf_llm_encoder.py` for text predictions, or `gemini_embedding_encoder.py` for representations).

## 1. Setup and Installation
First, we install the core `mseb` library directly from the source and configure the environment.

> ⚠️ **Note on Installation Warnings:** You may see red dependency conflict errors (such as `grpcio` or `pillow`) when running the install cell below. Colab comes pre-installed with many packages that might slightly conflict with MSEB's strict requirements. Because we are running a lightweight local tutorial and not a distributed pipeline, **you can safely ignore these warnings** as long as the cell finishes and prints the success message.

In [ ]:
# 1. Install MSEB, suppress the crcmod warning, and lock Pillow to prevent Torchvision conflicts
!pip install git+https://github.com/google-research/mseb.git crcmod "pillow<11.0" -q

import sys
from unittest.mock import MagicMock

# 2. Environment Mocking
# We mock the 'scann' vector search library here to avoid compiling heavy C++ 
# dependencies just for this lightweight tutorial. In a full benchmarking 
# environment running retrieval tasks, 'scann' would be natively installed.
mock_scann = MagicMock()
sys.modules['scann'] = mock_scann
sys.modules['scann.scann_ops'] = mock_scann
sys.modules['scann.scann_ops.py'] = mock_scann
sys.modules['scann.scann_ops.py.scann_ops_pybind'] = mock_scann

# 3. Import MSEB safely
import numpy as np
from typing import Sequence, Any
import tensorflow as tf

from mseb import encoder
from mseb.encoders import encoder_registry
from mseb import types

print("✅ MSEB Library loaded successfully!")

## 2. Building a Representation Encoder
Every model evaluated in MSEB must inherit from `encoder.MultiModalEncoder`. Because it is an Abstract Base Class (ABC), you are strictly required to implement three internal methods:
* `_setup(self)`: Used for lazy-loading your model weights. This prevents Out-Of-Memory errors when the registry first imports all available models.
* `_check_input_types(self, inputs)`: Validates that the incoming batch is the correct format.
* `_encode(self, inputs)`: The core engine where the actual generation happens. For representation models, it must return `types.TextEmbedding` or `types.SoundEmbedding`.

In [ ]:
class MyCustomEncoder(encoder.MultiModalEncoder):
    def __init__(self, embedding_dim: int = 768, **kwargs):
        # Pass any extra kwargs up to the base class
        super().__init__(**kwargs)
        self.embedding_dim = embedding_dim
        print(f"[MyCustomEncoder] Initialized with dimension {self.embedding_dim}")

    def _setup(self):
        # Initialize your real model (e.g., Hugging Face, PyTorch, etc.) here.
        # This is called lazily right before the first encode() call.
        print("[MyCustomEncoder] Running _setup() -> Loading model weights...")

    def _check_input_types(self, inputs: Sequence[Any]):
        # Optional: Add strict type checking here if you want to fail fast
        pass

    def _encode(self, inputs: Sequence[Any], **kwargs: Any) -> Sequence[Any]:
        print(f"[MyCustomEncoder] Executing internal _encode on {len(inputs)} items...")
        results = []
        
        for item in inputs:
            # 1. Generate a mock embedding of shape [1, D] 
            # (Replace this with your model's actual forward pass)
            vec = np.random.randn(1, self.embedding_dim).astype(np.float32)
            
            # 2. Package it into the strict MSEB Dataclasses.
            if isinstance(item, types.Text):
                # Text embeddings require character spans
                emb = types.TextEmbedding(
                    embedding=vec,
                    spans=np.array([[0, len(item.text)]]), 
                    context=item.context
                )
            elif isinstance(item, types.Sound):
                # Sound embeddings require timestamps in seconds
                emb = types.SoundEmbedding(
                    embedding=vec,
                    timestamps=np.array([[0.0, 1.0]]), 
                    context=item.context
                )
            else:
                raise ValueError(f"Unsupported input type: {type(item)}")
                
            results.append(emb)
            
        return results

## 3. Registering the Encoder
MSEB relies on an internal registry to fetch models dynamically. 
We must create an `EncoderMetadata` object and inject it into the registry so the evaluation runner can instantiate it.

In [ ]:
CUSTOM_ENCODER_NAME = "my_custom_encoder_v1"

# 1. Create the Metadata wrapper expected by MSEB
custom_metadata = encoder_registry.EncoderMetadata(
    name=CUSTOM_ENCODER_NAME,
    encoder=MyCustomEncoder,
    params=lambda: dict(embedding_dim=768),
    url="https://github.com/my-custom-model-url" # Optional documentation link
)

# 2. Inject into the active registry
encoder_registry._REGISTRY[CUSTOM_ENCODER_NAME] = custom_metadata

print(f"✅ Successfully registered: '{CUSTOM_ENCODER_NAME}'")

## 4. Verification and Execution
Let's prove the registration worked by asking MSEB to load our encoder by its string name and feed it strongly-typed dummy data. This simulates exactly how the benchmark runner will interact with your class.

In [ ]:
# Ask MSEB for the metadata from the registry
retrieved_metadata = encoder_registry.get_encoder_metadata(CUSTOM_ENCODER_NAME)
print(f"Retrieved Metadata for: {retrieved_metadata.name}")

# Instantiate the model using the MSEB lazy-loader
active_encoder = retrieved_metadata.load()

# Test Text Processing (Requires Context params!)
print("\n--- Testing Text Encoding ---")
test_texts = [
    types.Text(
        text="What is the capital of France?", 
        context=types.TextContextParams(id="test_q1")
    ), 
    types.Text(
        text="Translate to Spanish: Hello", 
        context=types.TextContextParams(id="test_q2")
    )
]
# The base class routes inputs to your _encode method automatically
text_outputs = active_encoder.encode(test_texts)

# Test Sound Processing (Requires Context params!)
print("\n--- Testing Sound Encoding ---")
dummy_sound = types.Sound(
    waveform=np.zeros(16000), 
    context=types.SoundContextParams(id="test_s1", sample_rate=16000, length=16000)
)
sound_outputs = active_encoder.encode([dummy_sound])

print("\n--- 📊 Results ---")
print(f"Successfully generated {len(text_outputs)} Text outputs!")
print(f"Text Output Shape: {text_outputs[0].embedding.shape}")
print(f"Successfully generated {len(sound_outputs)} Sound outputs!")
print(f"Sound Output Shape: {sound_outputs[0].embedding.shape}")
print("✅ Custom encoder is fully compatible with the MSEB pipeline!")

## 5. Bonus: Generative Encoders (LLMs)
If your model is an LLM designed to generate direct text answers (rather than embeddings), the setup is nearly identical. The only difference is that your `_encode` method should return `types.TextPrediction` objects instead of `types.TextEmbedding`.

In [ ]:
class MyCustomLLM(encoder.MultiModalEncoder):
    def _setup(self):
        print("\n[MyCustomLLM] Running _setup() -> Loading LLM weights...")

    def _check_input_types(self, inputs: Sequence[Any]):
        pass

    def _encode(self, inputs: Sequence[Any], **kwargs: Any) -> Sequence[Any]:
        results = []
        for item in inputs:
            # 1. Mock LLM generation (Replace with your model.generate())
            mock_answer = f"I am a generative AI, and you asked: '{item.text}'"
            
            # 2. Package into a TextPrediction object
            pred_context = types.PredictionContextParams(id=item.context.id)
            prediction = types.TextPrediction(
                prediction=mock_answer,
                context=pred_context
            )
            results.append(prediction)
        return results

# Register the LLM
encoder_registry._REGISTRY["my_custom_llm_v1"] = encoder_registry.EncoderMetadata(
    name="my_custom_llm_v1",
    encoder=MyCustomLLM,
    params=lambda: {}
)

# Test the LLM
active_llm = encoder_registry.get_encoder_metadata("my_custom_llm_v1").load()

llm_query = [types.Text(text="Who wrote Hamlet?", context=types.TextContextParams(id="llm_q1"))]
llm_outputs = active_llm.encode(llm_query)

print("\n--- 🤖 Generative LLM Results ---")
print(f"LLM Prediction: \"{llm_outputs[0].prediction}\"")
print("✅ Generative LLM is fully compatible with the MSEB pipeline!")